SymPy (https://docs.sympy.org/latest/guides/solving/index.html) can symbolically solve equations, differential equations, linear equations, nonlinear equations, matrix problems, inequalities, Diophantine equations, and evaluate integrals. SymPy can also solve numerically.
https://www.sympy.org/scipy-2017-codegen-tutorial/notebooks/01-intro-sympy.html
These examples show how SymPy:<br>
1. Solve an equation algebraically
2. Solve a system of equations algebraically
3. Solve one or a system of equations numerically
4. Solve an ordinary differential equation algebraically
5. Find the roots of a polynomial algebraically or numerically
6. Solve a matrix equation algebraically
7. Reduce one or a system of inequalities for a single variable algebraically
8. Solve a Diophantine equation algebraically

Notes:
SymPy has a function called solve() which is designed to find the solutions of an equation or system of equations, or the roots of a function. While a common, colloquial expression is, for example, “solve an integral,” in SymPy’s terminology it would be “evaluate an integral.” 

Solving Guidance<br>
These guidelines apply to many types of solving.

Numeric Solutions - The vast majority of arbitrary nonlinear equations have no closed-form solution. The classes of equations that are solvable are basically:

1. Linear equations
2. Polynomials, except where limited by the Abel-Ruffini theorem (learn more about solving polynomials using a GroebnerBasis)
3. Equations that can be solved by inverting some transcendental functions
4. Problems that can be transformed into the cases above (e.g., by turning trigonometric functions into polynomials)
5. A few other special cases that can be solved with something like the Lambert W function
6. Equations that you can decompose() via any of the above

SymPy may reflect that your equation has no solutions that can be expressed algebraically (symbolically), or that SymPy lacks an algorithm to find a closed-form solution that does exist, by returning an error such as NotImplementedError:

In [ ]:
from sympy import solve, cos
from sympy.abc import x
solve(cos(x) - x, x, dict=True)

# I) Solve an Equation Algebraically  (symbolically) 
There are two high-level functions to solve equations, solve() and solveset(). 

In [15]:
from sympy.abc import x, y
from sympy import solve
solve(x**2 - y, x, dict=True)

[{x: -sqrt(y)}, {x: sqrt(y)}]

In [11]:
from sympy import solveset
from sympy.abc import x, y
solveset(x**2 - y, x)

{-sqrt(y), sqrt(y)}

In [ ]:
from sympy import Eq, solve, solveset
from sympy.abc import x, y
eqn = Eq(x**2, y)
eqn
solutions = solve(eqn, x, dict=True)
print(solutions)
solutions_set = solveset(eqn, x)
print(solutions_set)
for solution_set in solutions_set:
    print(solution_set)

Restrict the Domain of Solutions
By default, SymPy will return solutions in the complex domain, which also includes purely real and imaginary values. Here, the first two solutions are real, and the last two are imaginary:

In [ ]:
import sympy
from sympy import Symbol, solve, solveset
x = Symbol('x')
solve(x**4 - 256, x, dict=True)
solveset(x**4 - 256, x)

Restrict returned solutions to real numbers, or another domain or range, the different solving functions use different methods.

For solve(), place an assumption on the symbol to be solved for

In [ ]:
from sympy import Symbol, solve
x = Symbol('x', real=True)
solve(x**4 - 256, x, dict=True)

or restrict the solutions with standard Python techniques for filtering a list such as a list comprehension:

In [ ]:
from sympy import Or, Symbol, solve
x = Symbol('x', real=True)
expr = (x-4)*(x-3)*(x-2)*(x-1)
solution = solve(expr, x)
print(solution)
solution_outside_2_3 = [v for v in solution if (v.is_real and Or(v<2,v>3))]
print(solution_outside_2_3)

In [ ]:
# For solveset(), limit the output domain in the function call by setting a domain

from sympy import S, solveset
from sympy.abc import x
solveset(x**4 - 256, x, domain=S.Reals)

# or by restricting returned solutions to any arbitrary set, including an interval:

from sympy import Interval, pi, sin, solveset
from sympy.abc import x
solveset(sin(x), x, Interval(-pi, pi))

In [ ]:
# if you restrict the solutions to a domain in which there are no solutions, solveset() will return the empty set, EmptySet:

from sympy import solveset, S
from sympy.abc import x
solveset(x**2 + 1, x, domain=S.Reals)


Explicitly Represent Infinite Sets of Possible Solutions
solveset() can represent infinite sets of possible solutions and express them in standard mathematical notation, for example  for every integer value of :

In [ ]:

from sympy import pprint, sin, solveset
from sympy.abc import x
solution = solveset(sin(x), x)
pprint(solution)

In [ ]:

# However, solve() will return only a finite number of solutions: solve() tries to return just enough solutions so that all (infinitely many) solutions can generated from the returned solutions by adding integer multiples of the periodicity() of the equation

from sympy import sin, solve
from sympy.calculus.util import periodicity
from sympy.abc import x
f = sin(x)
solve(f, x)
periodicity(f, x)

Use the Solution Result. Substitute Solutions From solve() Into an Expression
You can substitute solutions from solve() into an expression.

A common use case is finding the critical points and values for a function . At the critical points, the Derivative equals zero (or is undefined). You can then obtain the function values at those critical points by substituting the critical points back into the function using subs(). You can also tell if the critical point is a maxima or minima by substituting the values into the expression for the second derivative: a negative value indicates a maximum, and a positive value indicates a minimum.

In [ ]:
from sympy.abc import x
from sympy import solve, diff
f = x**3 + x**2 - x
derivative = diff(f, x)
critical_points = solve(derivative, x, dict=True)
print(critical_points)
point1, point2 = critical_points
print(f.subs(point1))
print(f.subs(point2))
curvature = diff(f, x, 2)
print(curvature.subs(point1))
print(curvature.subs(point2))

solveset() Solution Sets Cannot Necessarily Be Interrogated Programmatically
If solveset() returns a finite set (class FiniteSet), you can iterate through the solutions:

In [ ]:
from sympy import solveset
from sympy.abc import x, y
solution_set = solveset(x**2 - y, x)
print(solution_set)
solution_list = list(solution_set)
print(solution_list)

In [ ]:
# However, for more complex results, it may not be possible to list the solutions:

from sympy import S, solveset, symbols, intersection, sqrt
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
list(solution_set)

TypeError: The computation had not completed because of the undecidable set
membership is found in every candidates.

In this case, it is because, if  is negative, its square root would be imaginary rather than real and therefore outside the declared domain of the solution set. By declaring  to be real and positive, SymPy can determine that its square root is real, and thus resolve the intersection between the solutions and the set of real numbers:

In [ ]:
from sympy import S, Symbol, solveset
x = Symbol('x')
y = Symbol('y', real=True, positive=True)
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)

list(solution_set)

In [ ]:
# Alternatively, you can extract the sets from the solution set using args, then create a list from the set containing the symbolic solutions:

from sympy import S, solveset, symbols, intersection
x, y = symbols('x, y')
solution_set = solveset(x**2 - y, x, domain=S.Reals)
print(solution_set)
intersection({-sqrt(y), sqrt(y)}, domain=S.Reals)
solution_set_args = solution_set.args
print(solution_set.args)

list(solution_set_args[1])

Not All Equations Can Be Solved. Equations With No Closed-Form Solution
Some equations have no closed-form solution, in which case SymPy may return an empty set or give an error. For example, the following transcendental equation has no closed-form solution:

Equations Which Have a Closed-Form Solution, and SymPy Cannot Solve. 
It is also possible that there is an algebraic solution to your equation, and SymPy has not implemented an appropriate algorithm. 

In [ ]:
from sympy import cos, solve
from sympy.abc import x
solve(cos(x) - x, x, dict=True)

# II) Solve a System of Equations Algebraically

linear or nonlinear. For example, solving $x^2 + y = 2z, y = -4z$ for x and y (assuming z
is a constant or parameter) yields $\{(x = -\sqrt{6z}, y = -4z),$ ${(x =
\sqrt{6z}, y = -4z)\}}$.

- Some systems of equations cannot be solved algebraically (either at all or by SymPy), so you may have to [solve your system of equations numerically](solve-numerically.md) using {func}`~.nsolve` instead.

Whether your equations are linear or nonlinear, you can use {func}`~.solve`:

### Solve a System of Linear Equations Algebraically

In [7]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - 2*z, y + 4*z], [x, y], dict=True)
# solve([x + y - 2*z, y + 4*z], [x, y])

[{x: 6*z, y: -4*z}]

### Solve a System of Nonlinear Equations Algebraically

In [5]:
from sympy import solve
from sympy.abc import x, y, z
solve([x**2 + y - 2*z, y + 4*z], x, y, dict=True)

[{x: -sqrt(6)*sqrt(z), y: -4*z}, {x: sqrt(6)*sqrt(z), y: -4*z}]

Guidance : Refer to
[](solving-guidance.md#include-the-variable-to-be-solved-for-in-the-function-call)
and [](ensure-consistent-formatting-from-solve).

There are two methods below for containing solution results:
[dictionary](#solve-and-use-results-in-a-dictionary) or
[set](#solve-results-in-a-set). A dictionary is easier to interrogate
programmatically, so if you need to extract solutions using code, we recommend
the dictionary approach.

### Solve and Use Results in a Dictionary

Solve Into a Solution Given as a Dictionary

You can solve a system of equations for some variables (for example, $x$ and
$y$) leaving another symbol as a constant or parameter (for example, $z$). You
can specify the variables to solve for as multiple separate arguments, or as a
list (or tuple):

In [16]:
from sympy import solve
from sympy.abc import x, y, z
equations = [x**2 + y - 2*z, y + 4*z]
solutions = solve(equations, x, y, dict=True)
solutions

[{x: -sqrt(6)*sqrt(z), y: -4*z}, {x: sqrt(6)*sqrt(z), y: -4*z}]

Use a Solution Given as a Dictionary

You can then extract solutions by indexing (specifying in brackets) the solution
number, and then the symbol. For example `solutions[0][x]` gives the result for
`x` in the first solution:

In [ ]:
solutions[0][x]
solutions[0][y]

-4*z

Solve Results in a Set

To get a list of symbols and set of solutions, use `set=True` instead of
`dict=True`:

In [20]:
from sympy import solve
from sympy.abc import x, y, z
solve([x**2 + y - 2*z, y + 4*z], [x, y], set=True)
# ([x, y], {(-sqrt(6)*sqrt(z), -4*z), (sqrt(6)*sqrt(z), -4*z)})

([x, y], {(-sqrt(6)*sqrt(z), -4*z), (sqrt(6)*sqrt(z), -4*z)})


Options That Can Speed up {func}`~.solve`. Refer to [](options-that-can-speed-up-solve).

Not All Systems of Equations Can be Solved. Systems of Equations With no Solution
e.g., the following two systems have no solution because they reduce to `1 == 0`, so SymPy returns an empty list:

In [1]:
from sympy import solve
from sympy.abc import x, y
solve([x + y - 1, x + y], [x, y], dict=True)

[]

In [3]:

from sympy import solve
from sympy.abc import x, y, z
solve([x + y - (z + 1), x + y - z], [x, y], dict=True)

[]


The following system reduces to $z = 2z$, so it has no general solution, but it
could be satisfied if $z=0$. Note that {func}`~.solve` will not assume that
$z=0$, even though that is the only value of $z$ that makes the system of
equations consistent, because $z$ is a parameter rather than an unknown. That
is, {func}`~.solve` does not treat $z$ as an unknown because it is not in the
list of symbols specified as unknowns (`[x, y]`) and all such symbols are
treated like parameters with arbitrary value. Whether a symbol is treated as a
variable or a parameter is determined only by whether it is specified as a
symbol to solve for in {func}`~.solve`. There is no such distinction made when
creating the symbol using {func}`~.symbols` (or importing from {mod}`~.abc`).

In [4]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - z, x + y - 2*z], [x, y], dict=True)

[]

The following system is [overconstrained](https://en.wikipedia.org/wiki/Overdetermined_system), meaning there are more equations (three) than unknowns to be solved for (two, namely $x$ and $y$). It has no solution:

In [5]:
from sympy import solve
from sympy.abc import x, y, z
solve([x + y - z, x - (z + 1), 2*x - y], [x, y], dict=True)

[]

Note that some overconstrained systems do have solutions (for example, if an
equation is a linear combination of the others), in which case SymPy can solve
the overconstrained system.

Systems of Equations With no Closed-Form Solution

Some systems of equations cannot be solved algebraically, for example those
containing [transcendental equations](https://en.wikipedia.org/wiki/Transcendental_equation):


In [ ]:
from sympy import cos, solve
from sympy.abc import x, y, z
solve([x - y, cos(x) - y], [x, y], dict=True)

So you can use {func}`~.nsolve` to [find a numerical
solution](solve-numerically.md):

In [7]:
from sympy import cos, nsolve
from sympy.abc import x, y, z
nsolve([x - y, cos(x) - y], [x, y], [1,1])

Matrix([
[0.739085133215161],
[0.739085133215161]])

Equations Which Have a Closed-Form Solution, and SymPy Cannot Solve

It is also possible that there is an algebraic solution to your equation, and
SymPy has not implemented an appropriate algorithm. If SymPy returns an empty
set or list when you know there is a closed-form solution (indicating a bug in
SymPy), please post it on the [mailing list](https://groups.google.com/g/sympy),
or open an issue on [SymPy's GitHub page](https://github.com/sympy/sympy/issues). Until the issue is resolved, you can use a different method listed in [](#alternatives-to-consider).

# III) Solve One or a System of Equations Numerically
Solving numerically is useful if:

You only need a numeric solution, not a symbolic one

A closed-form solution is not available or is overly complicated; refer to When You Might Prefer a Numeric Solution

solve() and solveset() will not try to find a numeric solution, only a mathematically-exact symbolic solution. So if you want a numeric solution, use nsolve().

SymPy is designed for symbolic mathematics. If you do not need to do symbolic operations, then for numerical operations you can use another free and open-source package such as NumPy or SciPy which will be faster, work with arrays, and have more algorithms implemented. The main reasons to use SymPy (or its dependency mpmath) for numerical calculations are:

to do a simple numerical calculation within the context of a symbolic calculation using SymPy

if you need the arbitrary precision capabilities to get more digits of precision than you would get from float64.

Alternatives to Consider
SciPy’s scipy.optimize.fsolve() can solve a system of (non-linear) equations

NumPy’s numpy.linalg.solve() can solve a system of linear scalar equations

mpmath’s findroot(), which nsolve() calls and can pass parameters to
 


Example of Numerically Solving an Equation

In [ ]:
from sympy import cos, nsolve, Symbol
x = Symbol('x')
nsolve(cos(x) - x, x, 1)

0.739085133215161

Overdetermined systems of equations are supported.

Find Complex Roots of a Real Function
To solve for complex roots of real functions, specify a nonreal (either purely imaginary, or complex) initial point:

In [ ]:
from sympy import nsolve
from sympy.abc import x
nsolve(x**2 + 2, 1) # Real initial point returns no root

In [4]:
# Try another starting point or tweak arguments.
from sympy import I
nsolve(x**2 + 2, I) # Imaginary initial point returns a complex root

1.4142135623731*I

In [6]:
nsolve(x**2 + 2, 1 + I) # Complex initial point returns a complex root

1.4142135623731*I

Ensure the Root Found is in a Given Interval
It is not guaranteed that nsolve() will find the root closest to the initial point. Here, even though the root -1 is closer to the initial point of -0.1, nsolve() finds the root 1:

In [7]:
from sympy import nsolve
from sympy.abc import x
nsolve(x**2 - 1, -0.1)

1.00000000000000

You can ensure the root found is in a given interval, if such a root exists, using solver='bisect' by specifying the interval in a tuple. Here, specifying the interval (-10, 0) ensures that the root -1 is found:

In [8]:
from  sympy import nsolve
from sympy.abc import x
nsolve(x**2 - 1, (-10, 0), solver='bisect')

-1.00000000000000

Solve a System of Equations Numerically
To solve a system of multidimensional functions, supply a tuple of

functions (f1, f2)

variables to solve for (x1, x2)

starting values (-1, 1)

In [ ]:
from sympy import Symbol, nsolve
x1 = Symbol('x1')
x2 = Symbol('x2')
f1 = 3 * x1**2 - 2 * x2**2 - 1
f2 = x1**2 - 2 * x1 + x2**2 + 2 * x2 - 8
print(nsolve((f1, f2), (x1, x2), (-1, 1)))

#Increase Precision of the Solution
# You can increase the precision of the solution using prec:

from sympy import Symbol, nsolve
x1 = Symbol('x1')
x2 = Symbol('x2')
f1 = 3 * x1**2 - 2 * x2**2 - 1
f2 = x1**2 - 2 * x1 + x2**2 + 2 * x2 - 8
print(nsolve((f1, f2), (x1, x2), (-1, 1), prec=25))

Matrix([[-1.19287309935246], [1.27844411169911]])
Matrix([[-1.192873099352460791205211], [1.278444111699106966687122]])


Create a Function That Can Be Solved With SciPy
As noted above, SymPy focuses on symbolic computation and is not optimized for numerical calculations. If you need to make many calls to a numerical solver, it can be much faster to use a solver optimized for numerical calculations such as SciPy’s root_scalar(). A recommended workflow is:

use SymPy to generate (by symbolically simplifying or solving an equation) the mathematical expression

convert it to a lambda function using lambdify()

use a numerical library such as SciPy to generate numerical solutions

In [15]:
from sympy import simplify, cos, sin, lambdify
from sympy.abc import x, y
import scipy as sp 
# from scipy.optimize import root_scalar
expr = cos(x * (x + x**2)/(x*sin(y)**2 + x*cos(y)**2 + x))
simplify(expr) # 1. symbolically simplify expression
cos(x*(x + 1)/2)
lam_f = lambdify(x, cos(x*(x + 1)/2)) # 2. lambdify
sol = root_scalar(lam_f, bracket=[0, 2]) # 3. numerically solve using SciPy
sol.root

ModuleNotFoundError: No module named 'scipy'





Matrix([[-1.192873099352460791205211], [1.278444111699106966687122]])



1.3416277185114782
Use the Solution Result
Substitute the Result Into an Expression
The best practice is to use evalf() to substitute numerical values into expressions. The following code demonstrates that the numerical value is not an exact root because substituting it back into the expression produces a result slightly different from zero:

from sympy import cos, nsolve, Symbol
x = Symbol('x')
f = cos(x) - x
x_value = nsolve(f, x, 1); x_value
0.739085133215161
f.evalf(subs={x: x_value})
-5.12757857962640e-17
Using subs can give an incorrect result due to precision errors, here effectively rounding -5.12757857962640e-17 to zero:

f.subs(x, x_value)
0
When substituting in values, you can also leave some symbols as variables:

from sympy import cos, nsolve, Symbol
x = Symbol('x')
f = cos(x) - x
x_value = nsolve(f, x, 1); x_value
0.739085133215161
y = Symbol('y')
z = Symbol('z')
g = x * y**2
values = {x: x_value, y: 1}
(x + y - z).evalf(subs=values)
1.73908513321516 - z
Not all Equations Can be Solved
nsolve() is a numerical solving function, so it can often provide a solution for equations which cannot be solved algebraically.

Equations With no Solution
Some equations have no solution, in which case SymPy may return an error. For example, the equation 
 (exp(x) in SymPy) has no solution:

from sympy import nsolve, exp
from sympy.abc import x
nsolve(exp(x), x, 1, prec=20)
Traceback (most recent call last):
...
ValueError: Could not find root within given tolerance. (5.4877893607115270300540019e-18 > 1.6543612251060553497428174e-24)
Try another starting point or tweak arguments.